# ResistAI Quickstart

This notebook walks through the **ResistAI** drug-discovery SDK end-to-end:
protein structure prediction, drug screening, molecular docking, novel molecule
generation, and closed-loop optimization -- all powered by NVIDIA NIM biology
microservices.

**Target:** NDM-1 metallo-beta-lactamase, a key driver of carbapenem-resistant
Enterobacteriaceae (AMR threat level: critical).

## Setup

Set your API keys before running any cells.  
- **NVIDIA_API_KEY** -- get one free at [build.nvidia.com](https://build.nvidia.com)  
- **NEBIUS_API_KEY** -- get one at [nebius.com](https://nebius.com) (needed only for LLM analysis step)

In [ ]:
import os
os.environ["NVIDIA_API_KEY"] = "your-key-here"  # Get from build.nvidia.com
os.environ["NEBIUS_API_KEY"] = "your-key-here"   # Get from nebius.com

import sys
sys.path.insert(0, "..")

from src.sdk import Pipeline, Target
from src.models.nim_client import NIMClient

print("Imports OK")

## 1. Check Model Availability

Verify that the NVIDIA NIM biology endpoints are reachable with your API key.

In [ ]:
nim = NIMClient()
status = nim.health_check()

for endpoint, reachable in status.items():
    icon = "OK" if reachable else "UNREACHABLE"
    print(f"  {endpoint:15s} {icon}")

## 2. Create a Target

Define the protein target from its amino-acid sequence. The SDK also supports
`Target.from_uniprot("C7C422")` and `Target.from_pdb("3SPU")` for automatic
fetching.

In [ ]:
NDM1_SEQUENCE = (
    "MELPNIMHPVAKLSTALAAALMLSGCMPGEIRPTIGQQMETGDQRFGDLVFRQLAPNVWQ"
    "HTSYLDMPGFGAVASNGLIVRDGGRVLVVDTAWTDDQTAQILNWIKQEINLPVALAVVTH"
    "AHQDKMGGMDALHAAGIATYANALSNQLAPQEGMVAAQHSLTFAANGWVEPATAPNFGPLN"
    "VDSPIAGIAPHQFHEHGKIVYALGEHIERTARMAATFPKLIACVGGTLHEKTHPRHTLHAA"
    "VEGVSAALGHPIGDTIGHSLMTLAQNPDIGKLLHCAENQPAKLNDVNHAEQILSAAQKWT"
)

target = Target.from_sequence(NDM1_SEQUENCE, name="NDM-1")
print(target)

## 3. Build and Run Pipeline

Chain pipeline steps using a fluent API. Steps are **lazy** -- nothing runs
until `.run()` is called.

| Step | NIM Model | What it does |
|------|-----------|-------------|
| `fold()` | ESMFold | Predict 3D structure from sequence |
| `screen()` | Boltz-2 | Score binding of known antibiotics |
| `dock()` | DiffDock | Generate 3D binding poses |
| `generate()` | MolMIM | Design novel molecules |

In [ ]:
results = (
    Pipeline(target)
    .fold()
    .screen()
    .dock(top_k=3)
    .generate(n=3)
    .run(progress=True)
)

print(f"\nSteps completed: {results.steps_executed}")
print(f"Errors: {len(results.errors)}")

## 4. Explore Results

The `PipelineResults` object gives you typed access to every step's output.

In [ ]:
# -- Structure --
if results.structure:
    print(f"Structure predicted: {results.structure_length} chars, "
          f"{results.fold_elapsed_ms:.0f} ms")

# -- Screening hits --
print(f"\nScreening hits ({len(results.screening_hits)}):")
for i, hit in enumerate(results.screening_hits, 1):
    print(f"  {i}. {hit.drug_name:15s}  confidence={hit.confidence:.3f}  "
          f"iPTM={hit.iptm_score:.3f}")

# -- Docking poses --
print(f"\nDocking poses ({len(results.docking_poses)}):")
for i, pose in enumerate(results.docking_poses[:5], 1):
    print(f"  {i}. {pose.drug_name:15s}  confidence={pose.confidence:.3f}")

# -- Novel molecules --
print(f"\nNovel molecules ({len(results.novel_molecules)}):")
for i, mol in enumerate(results.novel_molecules, 1):
    print(f"  {i}. {mol.smiles[:60]}  score={mol.score:.3f} ({mol.score_type})")

## 5. Novel Molecule Optimization

The `MoleculeOptimizer` runs a closed-loop **Generate -> Score -> Select -> Repeat**
cycle. It uses MolMIM for generation and Boltz-2 for binding validation, optimizing
across multiple objectives (binding affinity, QED, Lipinski, toxicity).

In [ ]:
from src.sdk.optimizer import MoleculeOptimizer

optimizer = MoleculeOptimizer(
    target_sequence=NDM1_SEQUENCE,
    seed_smiles="OB(O)C1CCC(CC(=O)NC2CCNC2)CC1",  # Taniborbactam-like scaffold
    objectives=["binding", "qed", "lipinski", "toxicity"],
    max_iterations=2,
    molecules_per_iteration=3,
)

opt_results = optimizer.run()
print(opt_results.summary())

## 6. REST API

ResistAI also exposes a FastAPI server. Start it with:

```bash
uvicorn src.api.main:app --reload
```

Then call it from any HTTP client.

In [ ]:
import requests

API_BASE = "http://localhost:8000"

# Health check
resp = requests.get(f"{API_BASE}/health")
print("Health:", resp.json())

# Run a pipeline via the API
payload = {
    "sequence": NDM1_SEQUENCE,
    "name": "NDM-1",
    "steps": ["fold", "screen", "generate"],
    "generate_n": 3,
}
resp = requests.post(f"{API_BASE}/pipeline/run", json=payload)
print("Pipeline status:", resp.status_code)
print("Result keys:", list(resp.json().keys()))

## 7. Export Results

Pipeline results can be exported as JSON or a Markdown report.

In [ ]:
# JSON export
print(results.to_json()[:500], "...")

# Markdown report
report = results.to_report()
print("\n" + report[:1000], "...")

## Next Steps

- **Full pipeline:** Add `.validate()`, `.design_binder()`, and `.analyze()` steps
  for end-to-end discovery.
- **Custom drug library:** Pass your own compounds to `Pipeline.screen(drugs=[...])`.
- **PDB targets:** Use `Target.from_pdb("3SPU")` to load a known crystal structure
  and skip the fold step.
- **Longer optimization:** Increase `max_iterations` and `molecules_per_iteration`
  in `MoleculeOptimizer` for deeper exploration.
- **API docs:** Visit `http://localhost:8000/docs` after starting the server for
  interactive Swagger documentation.